In [1]:
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt

import albumentations as A
from albumentations.pytorch import ToTensorV2

import torch
from torch.utils.data import Dataset, DataLoader

import cv2
import numpy as np

In [2]:
train_transform = A.Compose([

    A.Resize(224,224),

    A.HorizontalFlip(p=0.5),

    A.RandomBrightnessContrast(
        brightness_limit=0.2,
        contrast_limit=0.2,
        p=0.5
    ),

    A.Rotate(
        limit=15,
        p=0.5
    ),

    A.GaussianBlur(
        blur_limit=(3,5),
        p=0.3
    ),

    A.CoarseDropout(
        max_holes=8,
        p=0.3
    ),

    A.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    ),

    ToTensorV2()
])

C:\Users\agarw\AppData\Local\Temp\ipykernel_13004\2358171628.py:23: UserWarning: Argument(s) 'max_holes' are not valid for transform CoarseDropout
  A.CoarseDropout(


In [3]:
val_transform = A.Compose([

    A.Resize(224,224),

    A.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    ),

    ToTensorV2()
])

In [4]:
class CattleDataset(Dataset):

    def __init__(self, root_dir, transform=None):

        self.root_dir = Path(root_dir)
        self.transform = transform

        self.image_paths = []
        self.labels = []

        self.class_names = sorted([
            folder.name
            for folder in self.root_dir.iterdir()
            if folder.is_dir()
        ])

        self.class_to_idx = {
            cls_name: idx
            for idx, cls_name in enumerate(self.class_names)
        }

        for cls_name in self.class_names:

            cls_folder = self.root_dir / cls_name

            for img_path in cls_folder.glob("*"):

                self.image_paths.append(img_path)
                self.labels.append(
                    self.class_to_idx[cls_name]
                )

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):

        img_path = self.image_paths[idx]

        image = cv2.imread(str(img_path))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        label = self.labels[idx]

        if self.transform:
            image = self.transform(image=image)["image"]

        return image, label

In [5]:
train_dataset = CattleDataset(
    root_dir="../datasets/identity_split/train",
    transform=train_transform
)

val_dataset = CattleDataset(
    root_dir="../datasets/identity_split/val",
    transform=val_transform
)

test_dataset = CattleDataset(
    root_dir="../datasets/identity_split/test",
    transform=val_transform
)

print("Train Images:", len(train_dataset))
print("Validation Images:", len(val_dataset))
print("Test Images:", len(test_dataset))

Train Images: 838
Validation Images: 341
Test Images: 139


In [6]:
train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=0
)

In [7]:
images, labels = next(iter(train_loader))

print(images.shape)
print(labels.shape)

torch.Size([16, 3, 224, 224])
torch.Size([16])


In [8]:
import torch
import torch.nn as nn
import torchvision.models as models

In [9]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(device)

cuda


In [10]:
class CattleEmbeddingModel(nn.Module):

    def __init__(self, embedding_size=512):

        super().__init__()

        self.backbone = models.resnet50(
            weights=models.ResNet50_Weights.DEFAULT
        )

        in_features = self.backbone.fc.in_features

        self.backbone.fc = nn.Identity()

        self.embedding = nn.Sequential(

            nn.Linear(in_features, 1024),

            nn.BatchNorm1d(1024),

            nn.ReLU(),

            nn.Dropout(0.3),

            nn.Linear(1024, embedding_size)

        )

    def forward(self, x):

        features = self.backbone(x)

        embeddings = self.embedding(features)

        return embeddings

In [11]:
model = CattleEmbeddingModel()

model = model.to(device)

In [12]:
images, labels = next(iter(train_loader))

images = images.to(device)

with torch.no_grad():

    embeddings = model(images)

print(embeddings.shape)

torch.Size([16, 512])


In [13]:
class CattleIdentificationModel(nn.Module):

    def __init__(self, num_classes):

        super().__init__()

        self.backbone = models.resnet50(
            weights=models.ResNet50_Weights.DEFAULT
        )

        in_features = self.backbone.fc.in_features

        self.backbone.fc = nn.Sequential(

            nn.Linear(in_features, 1024),

            nn.BatchNorm1d(1024),

            nn.ReLU(),

            nn.Dropout(0.3),

            nn.Linear(1024, 512),

            nn.ReLU(),

            nn.Linear(512, num_classes)
        )

    def forward(self, x):

        return self.backbone(x)

In [14]:
num_classes = len(train_dataset.class_names)

model = CattleIdentificationModel(
    num_classes=num_classes
)

model = model.to(device)

print("Classes:", num_classes)

Classes: 84


In [15]:
criterion = nn.CrossEntropyLoss()

In [16]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4
)

In [17]:
images, labels = next(iter(train_loader))

images = images.to(device)
labels = labels.to(device)

outputs = model(images)

loss = criterion(outputs, labels)

print(outputs.shape)
print(loss.item())

torch.Size([16, 84])
4.431826591491699


In [18]:
EPOCHS = 25

for epoch in range(EPOCHS):

    model.train()

    running_loss = 0.0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    epoch_loss = running_loss / len(train_loader)

    print(f"Epoch [{epoch+1}/{EPOCHS}] Loss: {epoch_loss:.4f}")

Epoch [1/25] Loss: 4.0248
Epoch [2/25] Loss: 2.8398
Epoch [3/25] Loss: 2.0210
Epoch [4/25] Loss: 1.4045
Epoch [5/25] Loss: 0.9802
Epoch [6/25] Loss: 0.6993
Epoch [7/25] Loss: 0.5764
Epoch [8/25] Loss: 0.5296
Epoch [9/25] Loss: 0.4920
Epoch [10/25] Loss: 0.4828
Epoch [11/25] Loss: 0.4638
Epoch [12/25] Loss: 0.4494
Epoch [13/25] Loss: 0.4549
Epoch [14/25] Loss: 0.4275
Epoch [15/25] Loss: 0.4463
Epoch [16/25] Loss: 0.4413
Epoch [17/25] Loss: 0.4219
Epoch [18/25] Loss: 0.4147
Epoch [19/25] Loss: 0.4177
Epoch [20/25] Loss: 0.4135
Epoch [21/25] Loss: 0.4034
Epoch [22/25] Loss: 0.4106
Epoch [23/25] Loss: 0.3937
Epoch [24/25] Loss: 0.4045
Epoch [25/25] Loss: 0.3901


In [19]:
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for images, labels in val_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)

        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total

print(f"Validation Accuracy: {accuracy:.2f}%")

Validation Accuracy: 0.29%


In [20]:
embedding_model = CattleEmbeddingModel()

embedding_model = embedding_model.to(device)

embedding_model.eval()

CattleEmbeddingModel(
  (backbone): ResNet(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): Bottleneck(
        (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (downsample): Sequenti

In [21]:
images, labels = next(iter(train_loader))

images = images.to(device)

with torch.no_grad():

    embeddings = embedding_model(images)

print(embeddings.shape)

torch.Size([16, 512])


In [22]:
import torch.nn.functional as F

similarity = F.cosine_similarity(
    embeddings[0].unsqueeze(0),
    embeddings[1].unsqueeze(0)
)

print(similarity.item())

0.6244127750396729


In [23]:
print(labels[0].item())
print(labels[1].item())

18
26


In [24]:
same_indices = []

for i in range(len(labels)):

    for j in range(i+1, len(labels)):

        if labels[i] == labels[j]:

            same_indices.append((i,j))

print(same_indices[:5])

[(0, 2), (3, 15), (7, 10)]


In [25]:
i, j = same_indices[0]

same_similarity = F.cosine_similarity(
    embeddings[i].unsqueeze(0),
    embeddings[j].unsqueeze(0)
)

print("Same Cow Similarity:", same_similarity.item())

Same Cow Similarity: 0.8134182691574097


In [26]:
embedding_database = []
label_database = []

embedding_model = CattleEmbeddingModel().to(device)

embedding_model.eval()

with torch.no_grad():

    for images, labels in train_loader:

        images = images.to(device)

        embeddings = embedding_model(images)

        embedding_database.append(
            embeddings.cpu()
        )

        label_database.append(
            labels
        )

embedding_database = torch.cat(
    embedding_database
)

label_database = torch.cat(
    label_database
)

print(embedding_database.shape)
print(label_database.shape)

torch.Size([838, 512])
torch.Size([838])


In [27]:
import torch.nn.functional as F

def identify_cow(query_embedding,
                 embedding_database,
                 label_database):

    similarities = F.cosine_similarity(
        query_embedding.unsqueeze(0),
        embedding_database
    )

    best_match_idx = torch.argmax(similarities)

    best_similarity = similarities[
        best_match_idx
    ].item()

    predicted_label = label_database[
        best_match_idx
    ].item()

    return predicted_label, best_similarity

In [28]:
query_embedding = embeddings[0].cpu()

predicted_label, similarity = identify_cow(
    query_embedding,
    embedding_database,
    label_database
)

print("Predicted Cow:", predicted_label)
print("Similarity:", similarity)

Predicted Cow: 78
Similarity: 0.9999999403953552


In [29]:
query_image, query_label = train_dataset[50]

query_tensor = query_image.unsqueeze(0).to(device)

embedding_model.eval()

with torch.no_grad():

    query_embedding = embedding_model(
        query_tensor
    )

query_embedding = query_embedding.cpu()

predicted_label, similarity = identify_cow(
    query_embedding.squeeze(0),
    embedding_database,
    label_database
)

print("Actual Label:", query_label)
print("Predicted Label:", predicted_label)
print("Similarity:", similarity)

Actual Label: 5
Predicted Label: 31
Similarity: 0.9396102428436279


In [30]:
class TrainedEmbeddingModel(nn.Module):

    def __init__(self, trained_model):

        super().__init__()

        self.features = nn.Sequential(
            *list(trained_model.backbone.children())[:-1]
        )

    def forward(self, x):

        x = self.features(x)

        x = x.view(x.size(0), -1)

        return x

In [31]:
trained_embedding_model = TrainedEmbeddingModel(model)

trained_embedding_model = trained_embedding_model.to(device)

trained_embedding_model.eval()

TrainedEmbeddingModel(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (4): Sequential(
      (0): Bottleneck(
        (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (downsample): Sequential(
          (

In [32]:
embedding_database = []
label_database = []

with torch.no_grad():

    for images, labels in train_loader:

        images = images.to(device)

        embeddings = trained_embedding_model(images)

        embedding_database.append(
            embeddings.cpu()
        )

        label_database.append(
            labels
        )

embedding_database = torch.cat(
    embedding_database
)

label_database = torch.cat(
    label_database
)

print(embedding_database.shape)
print(label_database.shape)

torch.Size([838, 2048])
torch.Size([838])


In [33]:
query_image, query_label = train_dataset[50]

query_tensor = query_image.unsqueeze(0).to(device)

with torch.no_grad():

    query_embedding = trained_embedding_model(
        query_tensor
    )

query_embedding = query_embedding.cpu()

predicted_label, similarity = identify_cow(
    query_embedding.squeeze(0),
    embedding_database,
    label_database
)

print("Actual Label:", query_label)
print("Predicted Label:", predicted_label)
print("Similarity:", similarity)

Actual Label: 5
Predicted Label: 31
Similarity: 0.8672187924385071


In [34]:
torch.save(
    model.state_dict(),
    "../models/cattle_resnet50.pth"
)

print("Model saved successfully.")

Model saved successfully.
